# LakeWatch DQ — Remediation / Requeue

Re-executes **one specific rule** (`rule_id`, mandatory) independently
of every other rule defined for its table. Use this after:
- Editing a rule (getting the UPDATE approved) and wanting its effect
  re-evaluated without waiting for the next full pipeline run, or
- Adding a brand-new rule to a table that already has data flowing
  through it, and wanting that new rule checked against both the
  already-quarantined backlog AND whatever is already sitting in the
  clean target table.

## What it does — two independent phases, both scoped to `rule_id` only

### Phase A — Re-validate already-quarantined records
For every `dq_bad_records` row in scope that this rule's own violations
produced:
1. Reconstruct the original row from `bad_record_data` (stored as JSON).
2. Re-apply **only this rule_id** (not every rule for the table).
3. If the row now passes, it is:
   - Merged (INSERT-only MERGE) into `<target_catalog>.<target_schema>.<table>`.
   - Marked `reprocess_status = 'REPROMOTED'` in `dq_bad_records` (the
     original audit row is kept; only status columns are updated).
4. Rows that still fail are left as `reprocess_status = 'STILL_FAILING'`.

### Phase B — Full-table rescan (NEW)
Runs `rule_id` against **every row currently in the target table**, not
just previously-quarantined ones. This is what lets a newly-added rule
catch violations in data that was written before the rule existed:
1. Query the full target table.
2. Evaluate `rule_id` against it.
3. Any violation is written to `dq_bad_records` (via the same audit
   path `dq_logs()` already uses) and logged to `dq_logs`.
4. If the rule is **blocking** (`enforcement_action != AUDIT_ONLY`),
   violating rows are additionally **deleted from the target table**
   (matched on `cdw_hash_key`, the standard stable identity column for
   cleaned tables in this pipeline). If the target table doesn't have
   `cdw_hash_key`, target-table cleanup is skipped with a clear warning
   — the violation is still fully audited either way.
5. If the rule is `AUDIT_ONLY`, rows stay in the target table — only
   the audit trail records the violation.

## Audit-trail distinction ("second time execution")
Every row this notebook writes to `dq_logs` is tagged
`execution_type = 'REQUEUE'` (the original pipeline's own runs are
always `'INITIAL'`), and `execution_run_no` is computed as one more
than the highest run number `dq_logs` already has for this exact
`rule_id` — so the very first time this rule is ever requeued shows as
run 2, the next requeue for the same rule shows as run 3, and so on.
This is a literal, queryable value, not just something inferred from
`batch_timestamp` ordering.

## enforcement_action awareness
Identical to `dq_executor.py`:
- `BLOCK_AND_AUDIT` (or NULL/unrecognised): a violation **blocks** promotion
  / triggers removal from the target table.
- `AUDIT_ONLY`: a violation is **logged** but the row is still promoted
  / left in the target table.

## Widgets — ALL mandatory except `violation_id`
| Widget | Effect |
|---|---|
| `table_name` | The table to requeue (mandatory) |
| `source_catalog_name` | Catalog the quarantined records originally came from (mandatory) |
| `source_schema_name` | Schema the quarantined records originally came from (mandatory) |
| `target_catalog_name` | Catalog holding the clean target table (mandatory) |
| `target_schema_name` | Schema holding the clean target table (mandatory) |
| `rule_id` | The ONE rule to re-execute independently (mandatory) |
| `since_date` | Only bad records quarantined on/after this date, YYYY-MM-DD (mandatory) |
| `violation_id` | Optional — scope Phase A to one specific violation batch |
| `dry_run` | `true` = preview only, nothing written (default: `true`) |

## Prerequisites
Run once on any existing installation (see `ddl/dq_tables_ddl.py`):
```sql
ALTER TABLE cdw_dev.domain_config.dq_bad_records
  ADD COLUMNS (reprocess_status STRING, reprocessed_at TIMESTAMP);
ALTER TABLE cdw_dev.domain_config.dq_logs
  ADD COLUMN IF NOT EXISTS execution_type STRING;
ALTER TABLE cdw_dev.domain_config.dq_logs
  ADD COLUMN IF NOT EXISTS execution_run_no INT;
```

In [0]:
import json
import sys
import os
import re
from datetime import datetime, timezone

from pyspark.sql import SparkSession, functions as F

# dq_framework lives one directory up from this notebook when it is deployed
# alongside dq_run.py; adjust FRAMEWORK_PATH if your layout differs.
FRAMEWORK_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                               "dq_framework")
sys.path.append(FRAMEWORK_PATH)

from dq_executor import (
    fetch_rules,
    apply_rule_based_casting,
    safe_cache,
    safe_unpersist,
    is_blocking_rule,
    ROW_ID_COLUMN,
    HASH_KEY_COLUMN,
    HOLD_BACK_SEVERITIES,
)
from query_builder import construct_query, normalize_type
from dq_logger import dq_logs, _write_log
from dq_error_logger import dq_error_logs

# ---------------------------------------------------------------------------
# Spark session
# ---------------------------------------------------------------------------
spark = SparkSession.builder \
    .appName("DQRequeue") \
    .enableHiveSupport() \
    .getOrCreate()

# ---------------------------------------------------------------------------
# Configuration (shared with dq_run.py via config.yaml)
# ---------------------------------------------------------------------------
import yaml

CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                            "config.yaml")
with open(CONFIG_PATH, "r") as _f:
    _config = yaml.safe_load(_f) or {}

CONFIG_CATALOG  = _config.get("config_catalog",  "dev_platform")
CONFIG_SCHEMA   = _config.get("config_schema",   "data_quality_config")
RESULT_CATALOG  = _config.get("result_catalog",  "dev_platform")
RESULT_SCHEMA   = _config.get("result_schema",   "data_quality_config")

BAD_RECORDS_TABLE = f"{RESULT_CATALOG}.{RESULT_SCHEMA}.dq_bad_records"
RULES_TABLE       = f"{CONFIG_CATALOG}.{CONFIG_SCHEMA}.dq_rules"
LOGS_TABLE        = f"{RESULT_CATALOG}.{RESULT_SCHEMA}.dq_logs"

# ---------------------------------------------------------------------------
# Widgets — ALL mandatory except violation_id.
# ---------------------------------------------------------------------------
dbutils.widgets.text("table_name",          "sm_iv00101")
dbutils.widgets.text("source_catalog_name", "dev_inventory")
dbutils.widgets.text("source_schema_name",  "bronze")
dbutils.widgets.text("target_catalog_name", "dev_data_quality")
dbutils.widgets.text("target_schema_name",  "bronze")
dbutils.widgets.text("rule_id",           "5c88b434-825d-45d6-9042-c7f1b1891fbf")
dbutils.widgets.text("since_date",          "2020-01-01")
dbutils.widgets.text("violation_id",        "dqv_20260909-074315-f8a3012c-d891-4de3-93a6-bdb48b44b26a")   # the ONLY optional widget
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

table_name          = dbutils.widgets.get("table_name").strip()
source_catalog_name = dbutils.widgets.get("source_catalog_name").strip()
source_schema_name  = dbutils.widgets.get("source_schema_name").strip()
target_catalog_name = dbutils.widgets.get("target_catalog_name").strip()
target_schema_name  = dbutils.widgets.get("target_schema_name").strip()
rule_id             = dbutils.widgets.get("rule_id").strip()
since_date          = dbutils.widgets.get("since_date").strip()
violation_id_flt     = dbutils.widgets.get("violation_id").strip() or None   # optional
dry_run             = dbutils.widgets.get("dry_run").strip().lower() == "true"

# FIX / REQUIREMENT: every widget is mandatory except violation_id. Fail
# fast and loudly, naming every missing one at once, instead of silently
# treating a blank widget as "no filter" (the previous behavior).
_missing = [name for name, val in {
    "table_name": table_name,
    "source_catalog_name": source_catalog_name,
    "source_schema_name": source_schema_name,
    "target_catalog_name": target_catalog_name,
    "target_schema_name": target_schema_name,
    "rule_id": rule_id,
    "since_date": since_date,
}.items() if not val]
if _missing:
    raise ValueError(
        f"The following widgets are mandatory and must not be blank: {', '.join(_missing)}. "
        f"(violation_id is the only optional widget.)"
    )

batch_timestamp = datetime.now(timezone.utc)

print("=" * 70)
print("DQ REQUEUE — starting")
print("=" * 70)
print(f"  table              = {source_catalog_name}.{source_schema_name}.{table_name}")
print(f"  target             = {target_catalog_name}.{target_schema_name}.{table_name}")
print(f"  rule_id            = {rule_id}   (this run executes ONLY this rule)")
print(f"  since_date         = {since_date}")
print(f"  violation_id       = {violation_id_flt or '(not scoped -- all violation batches for this rule)'}")
print(f"  dry_run            = {dry_run}")

DQ REQUEUE — starting
  table              = dev_inventory.bronze.sm_iv00101
  target             = dev_data_quality.bronze.sm_iv00101
  rule_id            = 34d36aa5-f16a-4c14-bb5e-d06c0508c05a   (this run executes ONLY this rule)
  since_date         = 2020-01-01
  violation_id       = dqv_20260909-074315-4a7c8bf0-93da-44eb-b4fa-a25b0bae3ee1
  dry_run            = True


### Step 1 — Load the ONE rule, and compute its execution_run_no

In [0]:
_, all_rules = fetch_rules(spark, CONFIG_CATALOG, CONFIG_SCHEMA,
                           source_catalog_name, source_schema_name)

# Scope down to exactly this rule_id, on exactly this table -- "execute this
# rule independently", not every active rule for the table.
matching_rules = [
    r for r in all_rules
    if r["rule_id"] == rule_id
    and r["catalog_name"].lower() == source_catalog_name.lower()
    and r["schema_name"].lower() == source_schema_name.lower()
    and r["table_name"].lower() == table_name.lower()
]

if not matching_rules:
    raise ValueError(
        f"No ACTIVE rule found with rule_id='{rule_id}' on "
        f"{source_catalog_name}.{source_schema_name}.{table_name}. "
        f"Check the rule_id, or confirm it's approved/active and not soft-deleted."
    )

rule = matching_rules[0]
ctype = rule.get("_ctype") or normalize_type(rule.get("condition_type", ""))
print(f"\nRule loaded: {rule_id} | condition_type={ctype} | "
      f"enforcement_action={rule.get('enforcement_action') or 'BLOCK_AND_AUDIT'} | "
      f"column={rule.get('column_name')}")

# execution_run_no: one more than the highest run number dq_logs already has
# for this rule_id. A rule that has only ever run via the main INITIAL
# pipeline (run 1) becomes run 2 on its first requeue, run 3 on its next, etc.
# This makes "this is the second time execution" a literal, queryable value.
_max_run_row = spark.sql(f"""
    SELECT MAX(execution_run_no) AS max_run
    FROM {LOGS_TABLE}
    WHERE rule_id = '{rule_id.replace("'", "''")}'
""").collect()
_prior_max_run = (_max_run_row[0]["max_run"] if _max_run_row else None) or 0
execution_run_no = _prior_max_run + 1
print(f"execution_run_no for this run  = {execution_run_no} "
      f"(prior highest recorded run: {_prior_max_run})")
if execution_run_no > 1:
    print(f"  -> This is a RE-execution (attempt #{execution_run_no}) of this rule, "
          f"distinctly tagged execution_type='REQUEUE' in dq_logs -- not "
          f"indistinguishable from the original run.")


Rule loaded: 34d36aa5-f16a-4c14-bb5e-d06c0508c05a | condition_type=MAPPING_CHECK | enforcement_action=AUDIT_ONLY | column=USCATVLS_6
execution_run_no for this run  = 2 (prior highest recorded run: 1)
  -> This is a RE-execution (attempt #2) of this rule, distinctly tagged execution_type='REQUEUE' in dq_logs -- not indistinguishable from the original run.


### Phase A — Re-validate already-quarantined records for this rule

In [0]:
print("\n" + "=" * 70)
print("PHASE A -- Re-validating already-quarantined dq_bad_records for this rule")
print("=" * 70)

def _bad_records_in_scope():
    """dq_bad_records rows in scope: this table, not already repromoted,
    and (via a dq_logs join) originally quarantined specifically by
    rule_id -- since violation_id can be shared across multiple rules'
    violations in the same batch, the rule_id match happens through a
    join, not a plain column filter."""
    filters = [
        f"LOWER(catalog_name) = LOWER('{source_catalog_name}')",
        f"LOWER(schema_name)  = LOWER('{source_schema_name}')",
        f"LOWER(table_name)   = LOWER('{table_name}')",
        "(reprocess_status IS NULL OR reprocess_status = 'STILL_FAILING')",
        f"batch_timestamp >= '{since_date}'",
    ]
    if violation_id_flt:
        safe_vid = violation_id_flt.replace("'", "''")
        filters.append(f"violation_id = '{safe_vid}'")
    where = " AND ".join(filters)
    df = spark.sql(f"SELECT * FROM {BAD_RECORDS_TABLE} WHERE {where}")

    safe_rid = rule_id.replace("'", "''")
    linked_ids = spark.sql(f"""
        SELECT DISTINCT violation_id
        FROM {LOGS_TABLE}
        WHERE rule_id = '{safe_rid}'
    """)
    return df.join(linked_ids, on="violation_id", how="inner")

bad_df = _bad_records_in_scope()
total_bad = bad_df.count()
print(f"Quarantined records in scope for this rule: {total_bad:,}")

phase_a_promote_rows   = []   # list of (br_id, row_dict)
phase_a_still_failing  = []   # list of br_id

if total_bad == 0:
    print("Nothing to re-validate in Phase A.")
else:
    sample_json = (bad_df
                   .select("bad_record_data")
                   .filter(F.col("bad_record_data").isNotNull())
                   .limit(1)
                   .collect())
    if not sample_json:
        print("Warning: No non-null bad_record_data rows -- skipping Phase A.")
    else:
        try:
            sample_dict = json.loads(sample_json[0]["bad_record_data"])
            inferred_schema = spark.createDataFrame([sample_dict]).schema
        except Exception as e:
            print(f"Warning: Could not infer schema from bad_record_data: {e} -- skipping Phase A.")
            inferred_schema = None

        if inferred_schema is not None:
            reconstructed = (bad_df
                             .withColumn("_rec", F.from_json(F.col("bad_record_data"), inferred_schema))
                             .select("br_id", "violation_id", F.col("_rec.*")))
            reconstructed = reconstructed.withColumn(
                ROW_ID_COLUMN, F.concat(F.lit("requeue_"), F.col("br_id")))
            reconstructed = safe_cache(reconstructed)

            reconstructed, _ = apply_rule_based_casting(reconstructed, [rule])

            REQUEUE_VIEW = "_dq_requeue_view"
            reconstructed.createOrReplaceTempView(REQUEUE_VIEW)

            fail_row_ids = set()
            if ctype in ("ROW_COUNT_TREND", "KPI_CHECK", "CUSTOM_RULE"):
                print(f"  Rule type {ctype} is table-level, not applicable to per-row "
                      f"re-validation -- Phase A treats all quarantined rows as still failing.")
                fail_row_ids = {row[ROW_ID_COLUMN] for row in reconstructed.select(ROW_ID_COLUMN).collect()}
            else:
                try:
                    query = construct_query(
                        spark, RESULT_CATALOG, RESULT_SCHEMA,
                        rule.get("pipeline_key"), rule.get("rule_id"),
                        "", "", REQUEUE_VIEW,
                        rule.get("column_name"), rule.get("condition_type"),
                        rule.get("condition_criteria"),
                        rule.get("condition_category"), batch_timestamp,
                        real_catalog=source_catalog_name, real_schema=source_schema_name,
                        real_table=table_name)

                    # FIX: do NOT use dq_logs() here -- it automatically
                    # quarantines any violation it finds as a BRAND NEW
                    # dq_bad_records row, which would duplicate the
                    # ORIGINAL bad-record row that this Phase already
                    # marks STILL_FAILING via its own UPDATE statement
                    # further down. Execute the query directly and write
                    # only the audit-log entry -- Phase A's own status
                    # tracking is the sole source of truth for
                    # dq_bad_records changes, so nothing gets duplicated.
                    violation_df = spark.sql(query)
                    violation_count = violation_df.count()
                    threshold = int(rule.get("violation_threshold") or 0)
                    status = "Fail" if violation_count > threshold else "Pass"
                    fc = 1 if violation_count > 0 else 0

                    fkeys_df, fkey_col = None, None
                    if violation_count > 0:
                        for cand in (ROW_ID_COLUMN, HASH_KEY_COLUMN):
                            if cand in violation_df.columns:
                                fkeys_df = violation_df.select(cand).distinct()
                                fkey_col = cand
                                break

                    _write_log(spark, RESULT_CATALOG, RESULT_SCHEMA,
                               rule_id=rule.get("rule_id"),
                               pipeline_key=rule.get("pipeline_key"),
                               catalog_name=source_catalog_name,
                               schema_name=source_schema_name,
                               table_name=table_name,
                               column_name=rule.get("column_name"),
                               condition_category=rule.get("condition_category"),
                               condition_type=ctype,
                               status=status,
                               violation_count=violation_count,
                               violation_id=None,
                               severity=rule.get("severity"),
                               batch_timestamp=batch_timestamp,
                               execution_type="REQUEUE",
                               execution_run_no=execution_run_no)

                    print(f"  dq_logs entry written: status={status} "
                          f"execution_type=REQUEUE execution_run_no={execution_run_no} "
                          f"(audit-only -- no new dq_bad_records row written here; "
                          f"see the STILL_FAILING update below for the ORIGINAL row instead)")

                    if fc > 0 and fkeys_df is not None and is_blocking_rule(rule):
                        if fkey_col == ROW_ID_COLUMN:
                            fail_row_ids = {r[0] for r in fkeys_df.collect()}
                        elif fkey_col == HASH_KEY_COLUMN:
                            fkeys = [r[0] for r in fkeys_df.collect()]
                            keys_lit = ",".join("'" + str(k).replace("'", "\\'") + "'" for k in fkeys)
                            sql = (f"SELECT {ROW_ID_COLUMN} FROM {REQUEUE_VIEW} "
                                   f"WHERE {HASH_KEY_COLUMN} IN ({keys_lit})")
                            fail_row_ids = {r[0] for r in spark.sql(sql).collect()}

                except Exception as e:
                    dq_error_logs(spark, RESULT_CATALOG, RESULT_SCHEMA,
                                  rule_id, rule.get("pipeline_key"), None,
                                  source_catalog_name, source_schema_name, table_name,
                                  f"Requeue Phase A rule evaluation error: {e}", batch_timestamp)
                    print(f"  Warning: Rule evaluation errored: {e}")

            spark.catalog.dropTempView(REQUEUE_VIEW)

            recon_collected = reconstructed.collect()
            safe_unpersist(reconstructed)

            for row in recon_collected:
                row_id = row[ROW_ID_COLUMN]
                br_id  = row["br_id"]
                if row_id in fail_row_ids:
                    phase_a_still_failing.append(br_id)
                else:
                    rd = {k: v for k, v in row.asDict().items()
                          if not k.startswith("_dq_") and k not in ("br_id", "violation_id")}
                    phase_a_promote_rows.append((br_id, rd))

            print(f"  To repromote  : {len(phase_a_promote_rows):,}")
            print(f"  Still failing : {len(phase_a_still_failing):,}")


PHASE A -- Re-validating already-quarantined dq_bad_records for this rule
Quarantined records in scope for this rule: 6,920
  dq_logs entry written: status=Fail execution_type=REQUEUE execution_run_no=2 (audit-only -- no new dq_bad_records row written here; see the STILL_FAILING update below for the ORIGINAL row instead)
  To repromote  : 6,920
  Still failing : 0


### Phase B — Full-table rescan (catches violations in rows this rule has never seen)

In [0]:
print("\n" + "=" * 70)
print("PHASE B -- Rescanning the full target table for NEW violations of this rule")
print("=" * 70)

target_table_fq = f"{target_catalog_name}.{target_schema_name}.{table_name}"
phase_b_violation_count = 0
phase_b_removed_from_target = 0
phase_b_delete_skipped_no_key = False

try:
    spark.sql(f"DESCRIBE TABLE {target_table_fq}")
    target_exists = True
except Exception:
    target_exists = False
    print(f"  Target table {target_table_fq} does not exist yet -- nothing to rescan.")

if target_exists:
    if ctype in ("ROW_COUNT_TREND", "KPI_CHECK", "CUSTOM_RULE"):
        print(f"  Rule type {ctype} is table-level, not a per-row check -- "
              f"Phase B rescan does not apply to this rule type.")
    else:
        target_df = spark.sql(f"SELECT * FROM {target_table_fq}")
        target_row_count = target_df.count()
        print(f"  Rows currently in target table: {target_row_count:,}")

        target_df, _ = apply_rule_based_casting(target_df, [rule])
        target_df = safe_cache(target_df)

        RESCAN_VIEW = "_dq_requeue_rescan_view"
        target_df.createOrReplaceTempView(RESCAN_VIEW)

        try:
            query = construct_query(
                spark, RESULT_CATALOG, RESULT_SCHEMA,
                rule.get("pipeline_key"), rule.get("rule_id"),
                "", "", RESCAN_VIEW,
                rule.get("column_name"), rule.get("condition_type"),
                rule.get("condition_criteria"),
                rule.get("condition_category"), batch_timestamp,
                real_catalog=target_catalog_name, real_schema=target_schema_name,
                real_table=table_name)

            sc, fc, sev, vid, fkeys_df, fkey_col = dq_logs(
                spark, RESULT_CATALOG, RESULT_SCHEMA,
                rule.get("rule_id"), query, rule.get("pipeline_key"),
                target_catalog_name, target_schema_name, table_name,
                rule.get("column_name"),
                rule.get("condition_category"), ctype,
                rule.get("severity"), batch_timestamp,
                int(rule.get("violation_threshold") or 0),
                execution_type="REQUEUE", execution_run_no=execution_run_no)

            print(f"  dq_logs entry written: status={'Fail' if fc else 'Pass'} "
                  f"execution_type=REQUEUE execution_run_no={execution_run_no}")

            if fc > 0:
                # violation_count isn't directly returned by dq_logs -- recover
                # it from fkeys_df's row count as the best available count.
                phase_b_violation_count = fkeys_df.count() if fkeys_df is not None else 0
                print(f"  NEW violations found in target table: {phase_b_violation_count:,} "
                      f"(already written to dq_bad_records by dq_logs())")

                if is_blocking_rule(rule):
                    if fkey_col == HASH_KEY_COLUMN and fkeys_df is not None and not dry_run:
                        fkeys = [r[0] for r in fkeys_df.collect()]
                        keys_lit = ",".join("'" + str(k).replace("'", "\\'") + "'" for k in fkeys)
                        spark.sql(f"""
                            DELETE FROM {target_table_fq}
                            WHERE {HASH_KEY_COLUMN} IN ({keys_lit})
                        """)
                        phase_b_removed_from_target = len(fkeys)
                        print(f"  Removed {phase_b_removed_from_target:,} violating rows from "
                              f"{target_table_fq} (rule is blocking: enforcement_action="
                              f"{rule.get('enforcement_action') or 'BLOCK_AND_AUDIT'})")
                    elif fkey_col != HASH_KEY_COLUMN:
                        phase_b_delete_skipped_no_key = True
                        print(f"  Warning: Target table has no '{HASH_KEY_COLUMN}' column to safely "
                              f"match rows for deletion -- violations are fully audited in "
                              f"dq_logs/dq_bad_records, but could NOT be automatically removed "
                              f"from {target_table_fq}. Remove manually if needed.")
                    elif dry_run:
                        print(f"  DRY RUN -- would remove {fkeys_df.count() if fkeys_df is not None else 0:,} "
                              f"rows from {target_table_fq} (rule is blocking).")
                else:
                    print(f"  Rule is AUDIT_ONLY (enforcement_action="
                          f"{rule.get('enforcement_action')}) -- rows stay in "
                          f"{target_table_fq}, violation is logged only.")
            else:
                print(f"  No new violations found -- every row in the target table currently "
                      f"passes this rule.")

        except Exception as e:
            dq_error_logs(spark, RESULT_CATALOG, RESULT_SCHEMA,
                          rule_id, rule.get("pipeline_key"), None,
                          target_catalog_name, target_schema_name, table_name,
                          f"Requeue Phase B rescan error: {e}", batch_timestamp)
            print(f"  Warning: Phase B rescan errored: {e}")

        spark.catalog.dropTempView(RESCAN_VIEW)
        safe_unpersist(target_df)


PHASE B -- Rescanning the full target table for NEW violations of this rule
  Rows currently in target table: 71,328
🚨 Violations 71328 >10% over average (26082.67). Severity escalated to CRITICAL.
🚨 Severity escalated to CRITICAL due to abnormal violation count: 71328
  dq_logs entry written: status=Fail execution_type=REQUEUE execution_run_no=2
  NEW violations found in target table: 71,328 (already written to dq_bad_records by dq_logs())
  Rule is AUDIT_ONLY (enforcement_action=AUDIT_ONLY) -- rows stay in dev_data_quality.bronze.sm_iv00101, violation is logged only.


### Preview (Phase A only — Phase B already wrote its changes above)

In [0]:
preview_schema = "br_id STRING, outcome STRING"
_preview_rows = ([(br, "REPROMOTE") for br, _ in phase_a_promote_rows] +
                  [(br, "STILL_FAILING") for br in phase_a_still_failing])
if _preview_rows:
    preview_df = spark.createDataFrame(_preview_rows, schema=preview_schema)
    display(preview_df.orderBy("outcome"))
else:
    print("No Phase A records to preview.")

print(f"\nPhase A summary -- to repromote: {len(phase_a_promote_rows):,} | "
      f"still failing: {len(phase_a_still_failing):,}")
print(f"Phase B summary -- new violations found: {phase_b_violation_count:,} | "
      f"removed from target: {phase_b_removed_from_target:,}")

if dry_run:
    print("\nDRY RUN -- Phase A changes not written (Phase B already dry-ran its own "
          "target-table delete above). Set dry_run=false to apply everything.")
    dbutils.notebook.exit(
        f"DRY RUN: Phase A would repromote {len(phase_a_promote_rows)}, "
        f"still failing {len(phase_a_still_failing)}; "
        f"Phase B found {phase_b_violation_count} new violations")

### Step 2 — Write Phase A changes (only when dry_run = false)

In [0]:
promoted_count = 0

if phase_a_promote_rows:
    br_ids_to_promote = [br for br, _ in phase_a_promote_rows]
    row_dicts         = [rd for _, rd in phase_a_promote_rows]

    try:
        promote_df = spark.createDataFrame(row_dicts)
        promote_df = safe_cache(promote_df)

        if HASH_KEY_COLUMN in promote_df.columns:
            merge_cond = f"target.`{HASH_KEY_COLUMN}` = source.`{HASH_KEY_COLUMN}`"
        else:
            shared_cols = [c for c in promote_df.columns if not c.startswith("_dq_")]
            merge_cond  = " AND ".join(f"target.`{c}` <=> source.`{c}`" for c in shared_cols)

        try:
            spark.sql(f"DESCRIBE TABLE {target_table_fq}")
            table_existed = True
        except Exception:
            table_existed = False

        if not table_existed:
            promote_df.write.format("delta").mode("overwrite") \
                .option("overwriteSchema", "true").saveAsTable(target_table_fq)
            promoted_count = len(row_dicts)
            print(f"Created and populated {target_table_fq} ({promoted_count:,} rows)")
        else:
            promote_df.createOrReplaceTempView("_requeue_promote")
            spark.sql(f"""
                MERGE INTO {target_table_fq} AS target
                USING _requeue_promote AS source
                ON {merge_cond}
                WHEN NOT MATCHED THEN INSERT *
            """)
            spark.catalog.dropTempView("_requeue_promote")
            promoted_count = len(row_dicts)
            print(f"Merged {promoted_count:,} Phase A rows -> {target_table_fq}")

        safe_unpersist(promote_df)

        # Mark these bad records REPROMOTED
        update_ts_str = batch_timestamp.strftime("%Y-%m-%d %H:%M:%S")
        CHUNK = 500
        for i in range(0, len(br_ids_to_promote), CHUNK):
            chunk = br_ids_to_promote[i:i + CHUNK]
            ids_lit = ", ".join(f"'{bid}'" for bid in chunk)
            spark.sql(f"""
                UPDATE {BAD_RECORDS_TABLE}
                SET    reprocess_status = 'REPROMOTED',
                       reprocessed_at   = TIMESTAMP '{update_ts_str}'
                WHERE  br_id IN ({ids_lit})
            """)
        print(f"Marked {len(br_ids_to_promote):,} dq_bad_records rows REPROMOTED")

    except Exception as e:
        dq_error_logs(spark, RESULT_CATALOG, RESULT_SCHEMA, rule_id,
                      rule.get("pipeline_key"), None,
                      source_catalog_name, source_schema_name, table_name,
                      f"Requeue Phase A promotion error: {e}", batch_timestamp)
        print(f"Error promoting Phase A rows to {target_table_fq}: {e}")

if phase_a_still_failing:
    update_ts_str = batch_timestamp.strftime("%Y-%m-%d %H:%M:%S")
    CHUNK = 500
    for i in range(0, len(phase_a_still_failing), CHUNK):
        chunk = phase_a_still_failing[i:i + CHUNK]
        ids_lit = ", ".join(f"'{bid}'" for bid in chunk)
        spark.sql(f"""
            UPDATE {BAD_RECORDS_TABLE}
            SET    reprocess_status = 'STILL_FAILING',
                   reprocessed_at   = TIMESTAMP '{update_ts_str}'
            WHERE  br_id IN ({ids_lit})
        """)
    print(f"Marked {len(phase_a_still_failing):,} dq_bad_records rows STILL_FAILING")

# Phase A summary entry in dq_logs (in addition to the per-rule entries
# already written above by dq_logs() itself during evaluation)
if phase_a_promote_rows or phase_a_still_failing:
    _write_log(spark, RESULT_CATALOG, RESULT_SCHEMA,
               rule_id=rule_id,
               pipeline_key=rule.get("pipeline_key"),
               catalog_name=source_catalog_name, schema_name=source_schema_name,
               table_name=table_name,
               column_name=rule.get("column_name"),
               condition_category="REQUEUE_SUMMARY",
               condition_type="REQUEUE_SUMMARY",
               status="Pass" if not phase_a_still_failing else "Fail",
               violation_count=len(phase_a_still_failing),
               violation_id=violation_id_flt,
               severity="Info",
               batch_timestamp=batch_timestamp,
               execution_type="REQUEUE",
               execution_run_no=execution_run_no)
    print("Phase A REQUEUE_SUMMARY entry written to dq_logs.")

Merged 6,920 Phase A rows -> dev_data_quality.bronze.sm_iv00101
Marked 6,920 dq_bad_records rows REPROMOTED
Phase A REQUEUE_SUMMARY entry written to dq_logs.


### Step 3 — Final summary

In [0]:
print("\n" + "=" * 70)
print("DQ REQUEUE COMPLETE")
print("=" * 70)
print(f"  rule_id                          : {rule_id}")
print(f"  execution_type / execution_run_no: REQUEUE / {execution_run_no}")
print(f"  --- Phase A (already-quarantined records) ---")
print(f"  Records repromoted to target     : {promoted_count:,}")
print(f"  Records still failing            : {len(phase_a_still_failing):,}")
print(f"  --- Phase B (full-table rescan)  ---")
print(f"  New violations found in target   : {phase_b_violation_count:,}")
print(f"  Rows removed from target table   : {phase_b_removed_from_target:,}"
      + ("  (skipped -- no cdw_hash_key column)" if phase_b_delete_skipped_no_key else ""))
print(f"  --- Where to look ---")
print(f"  Target (clean) table             : {target_table_fq}")
print(f"  Audit trail (this run's entries) : SELECT * FROM {LOGS_TABLE} "
      f"WHERE rule_id = '{rule_id}' AND execution_run_no = {execution_run_no}")
print(f"  Quarantined records              : SELECT * FROM {BAD_RECORDS_TABLE} "
      f"WHERE table_name = '{table_name}'")
print("=" * 70)


DQ REQUEUE COMPLETE
  rule_id                          : 34d36aa5-f16a-4c14-bb5e-d06c0508c05a
  execution_type / execution_run_no: REQUEUE / 2
  --- Phase A (already-quarantined records) ---
  Records repromoted to target     : 6,920
  Records still failing            : 0
  --- Phase B (full-table rescan)  ---
  New violations found in target   : 71,328
  Rows removed from target table   : 0
  --- Where to look ---
  Target (clean) table             : dev_data_quality.bronze.sm_iv00101
  Audit trail (this run's entries) : SELECT * FROM dev_platform.data_quality_config.dq_logs WHERE rule_id = '34d36aa5-f16a-4c14-bb5e-d06c0508c05a' AND execution_run_no = 2
  Quarantined records              : SELECT * FROM dev_platform.data_quality_config.dq_bad_records WHERE table_name = 'sm_iv00101'
